# Build the FFHQ real photos

Crops a fixed sample of FFHQ (Karras et al., CVPR 2019) with the same RetinaFace pipeline as `build_dataset.ipynb`, and uploads it as a **private** Hugging Face dataset. Every crop is bona fide; the `lab_ffpp_real_ffhq` protocol adds them to the training side, to test whether more kinds of real photo stop the detectors flagging real photos from sources they never saw (DiffusionFace's CelebA-HQ, `docs/results.md`).

**Source:** the 256 px copy on the Hub, `bitmind/ffhq-256` (16 parquet files, 7.6 GB, image names `00000.png` to `69999.png`). FFHQ is released by NVIDIA under CC BY-NC-SA 4.0 for non-commercial use; the images keep their Flickr licenses.

**Sample:** `ffhq.sampled` keeps about `KEEP` in every `OUT_OF` names, about 30,000 photos, as many as the real crops on `lab_ffpp_real`'s training side.

**Runtime:** Colab Pro with a GPU. **Needs** the Colab secret `HF_TOKEN` with write access, for the upload cell.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-ffhq"  # private
SOURCE_REPO = "bitmind/ffhq-256"
BUILD_DIR = "/content/build_ffhq"
KEEP, OUT_OF = 3, 7

### Installation

Same as `build_dataset.ipynb`: `onnxruntime-gpu` and the ONNX backend, so RetinaFace runs on the GPU.

In [ ]:
%%capture
!pip install -q "face-decision-lab[build] @ {REPO_SOURCE}"
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu

In [ ]:
import os
os.environ["DEEPFACE_BACKEND_ENGINE"] = "onnx"

import onnxruntime as ort
providers = ort.get_available_providers()
print("ONNX Runtime providers:", providers)
assert "CUDAExecutionProvider" in providers, "onnxruntime-gpu cannot see the GPU; RetinaFace would run on the CPU"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Build

One parquet file at a time, deleted after cropping. Resumable, like the main build: a rerun skips every image already in `processed.jsonl`.

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm
from face_decision_lab.build import DatasetBuilder
from face_decision_lab.faces import RetinaFaceDetector
from face_decision_lab.sources import ffhq

def rows(path):
    for batch in pq.ParquetFile(path).iter_batches(batch_size = 256, columns = ["image"]):
        for image in batch.column("image").to_pylist():
            if ffhq.sampled(image["path"], KEEP, OUT_OF):
                yield ffhq.parse_name(image["path"]), image["bytes"]

files = sorted(f for f in HfApi().list_repo_files(SOURCE_REPO, repo_type = "dataset") if f.endswith(".parquet"))
builder = DatasetBuilder(BUILD_DIR, RetinaFaceDetector())
for name in tqdm(files, desc = "parquet files"):
    path = Path(hf_hub_download(SOURCE_REPO, name, repo_type = "dataset", local_dir = "/content/ffhq_parquet"))
    added = builder.add_images(rows(path))
    builder.flush()
    path.unlink()
    print(f"{name}: {added} images")
print(len(builder.samples), "crops")

### Check detection outcomes

In [ ]:
for generator in sorted(builder.report.counts):
    counts = builder.report.counts[generator]
    print(f"{generator:28s} saved {builder.report.saved_share(generator):6.1%}  {counts}")

### Upload as a private dataset

In [ ]:
from face_decision_lab.hub import push_build

print(push_build(BUILD_DIR, HF_DATASET))